<a href="https://colab.research.google.com/github/tianlangxingaerfa/Python/blob/main/B_V_from_spectra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
from astropy.io import fits
import math

# PREVIOUS DATA
b_band_data = [(3600, 0.0), (3700, 0.030), (3800, 0.134), (3900, 0.567), (4000, 0.920), (4100, 0.978), (4200, 1.000), (4300, 0.978), (4400, 0.935), (4500, 0.853), (4600, 0.740), (4700, 0.640), (4800, 0.536), (4900, 0.424), (5000, 0.325), (5100, 0.235), (5200, 0.150), (5300, 0.095), (5400, 0.043), (5500, 0.009), (5600, 0.0)]
v_band_data = [(4700, 0.000), (4800, 0.030), (4900, 0.163), (5000, 0.458), (5100, 0.780), (5200, 0.967), (5300, 1.000), (5400, 0.973), (5500, 0.898), (5600, 0.792), (5700, 0.684), (5800, 0.574), (5900, 0.461), (6000, 0.359), (6100, 0.270), (6200, 0.197), (6300, 0.135), (6400, 0.081), (6500, 0.045), (6600, 0.025), (6700, 0.017), (6800, 0.013), (6900, 0.009), (7000, 0.000)]

def interpolate_transmission(wav_angstrom, band_data):
    if wav_angstrom <= band_data[0][0] or wav_angstrom >= band_data[-1][0]:
        return 0.0
    for i in range(len(band_data) - 1):
        x0, y0 = band_data[i]
        x1, y1 = band_data[i+1]
        if x0 <= wav_angstrom <= x1:
            return y0 + (wav_angstrom - x0) * (y1 - y0) / (x1 - x0)
    return 0.0

def calculate_star_color(fits_file, zero_point):
    try:
        with fits.open(fits_file) as hdul:
            data = hdul[0].data
            header = hdul[0].header

            # Wavelength reconstruction
            start_wav = header['CRVAL1']
            step_wav = header.get('CDELT1') or header.get('CD1_1')
            wavelengths = start_wav + np.arange(len(data)) * step_wav

            flux_B = 0
            flux_V = 0

            for i in range(len(wavelengths)):
                wav = wavelengths[i]
                f_lambda = data[i]

                flux_B += f_lambda * interpolate_transmission(wav, b_band_data)
                flux_V += f_lambda * interpolate_transmission(wav, v_band_data)

            if flux_B <= 0 or flux_V <= 0:
                return None

            b_v_calc = -2.5 * math.log10(flux_B / flux_V) + zero_point
            return b_v_calc
    except Exception as e:
        print(f"Error processing {fits_file}: {e}")
        return None

# Calculated Zero Point from Planck
zp = 0.5053

# Star List and SIMBAD Values
stars_files = [
    "stelib_spec_fits_HD102212_moy.fits",
    "stelib_spec_fits_HD120136_moy.fits",
    "stelib_spec_fits_HD123299_moy.fits",
    "stelib_spec_fits_HD147394_moy.fits",
    "stelib_spec_fits_HD184927_moy.fits"
]

simbad_data = {
    "HD102212": 1.5,
    "HD120136": 0.49,
    "HD123299": -0.04,
    "HD147394": -0.14,
    "HD184927": -0.15
}

# Comparison Table
print(f"{'Star ID':<10} | {'Calculated (B-V)':<16} | {'SIMBAD (B-V)':<12} | {'Diff'}")
print("-" * 60)

for file_name in stars_files:
    # Extract HD number from filename for dictionary lookup
    hd_id = file_name.split('_')[3] # Result like "HD102212"

    b_v_calc = calculate_star_color(file_name, zp)
    b_v_simbad = simbad_data.get(hd_id)

    if b_v_calc is not None:
        diff = b_v_calc - b_v_simbad
        print(f"{hd_id:<10} | {b_v_calc:<16.4f} | {b_v_simbad:<12.2f} | {diff:+.4f}")
    else:
        print(f"{hd_id:<10} | {'Error':<16} | {b_v_simbad:<12.2f} | N/A")

Star ID    | Calculated (B-V) | SIMBAD (B-V) | Diff
------------------------------------------------------------
HD102212   | 1.2964           | 1.50         | -0.2036
HD120136   | 0.3392           | 0.49         | -0.1508
HD123299   | -0.2565          | -0.04        | -0.2165
HD147394   | -0.2402          | -0.14        | -0.1002
HD184927   | 1.0052           | -0.15        | +1.1552
